# Reviewer checks run by scripts: negative controls, discretisation sensitivity, runtime

Summarises `negative_controls.tsv` (Reviewer 1, comment 7: label permutation and within-class feature shuffling, LOCO fold-0 training sets), `discretisation_sensitivity.tsv` (Reviewer 1, comment 10: cut range and number of bins) and `runtime.tsv` (Reviewer 1, comment 9). The scripts `run_*.py` in this folder produce these files.


In [ ]:
import os, pandas as pd
numbers = []
def note(claim, value, source): numbers.append({'section': 'R5', 'claim': claim, 'value': value, 'source': source}); print(f'{claim}: {value}')
if os.path.exists('negative_controls.tsv'):
    nc = pd.read_csv('negative_controls.tsv', sep='\t')
    s = nc.groupby(['data_type', 'control']).agg(runs=('permutation', 'count'), pairs_mean=('unordered_pairs_passing', 'mean'), pairs_max=('unordered_pairs_passing', 'max'),
                                                 sig2d_mean=('features_2D_significant', 'mean'), max_total_IG=('max_total_IG', 'max')).round(1)
    display(s)
    for dt in ('taxa', 'functions', 'combined'):
        real = nc[(nc.data_type == dt) & (nc.control == 'real')].unordered_pairs_passing.iloc[0]
        for k in ('label_perm', 'within_class'):
            g = nc[(nc.data_type == dt) & (nc.control == k)]
            note(f'{dt}: synergistic pairs with real labels vs {k} (mean, max over {len(g)} permutations)', f'{real:,} vs {g.unordered_pairs_passing.mean():.1f} (max {g.unordered_pairs_passing.max()})', 'R5 negative_controls.tsv')


In [ ]:
if os.path.exists('discretisation_sensitivity.tsv'):
    ds = pd.read_csv('discretisation_sensitivity.tsv', sep='\t'); display(ds)
    for _, r in ds.iterrows():
        note(f'discretisation {r.config}: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG', f'{r.unordered_pairs_passing:,} / {r.jaccard_all_vs_default} / {r.jaccard_top2pct_vs_default} / {r.spearman_total_IG_shared_pairs}', 'R5 discretisation_sensitivity.tsv')
if os.path.exists('runtime.tsv'):
    rt = pd.read_csv('runtime.tsv', sep='\t'); display(rt)
    for _, r in rt.iterrows():
        note(f'runtime one MDFS run, {r.data_type} ({r.n_samples} x {r.n_features}, {r.candidate_pairs:,} candidate pairs)', f'{r.wall_clock_s:.0f} s, peak {r.peak_RSS_GB} GB', 'R5 runtime.tsv')
    print(rt.machine.iloc[0])
pd.DataFrame(numbers).to_csv('r5_numbers_scripts.tsv', sep='\t', index=False)
